# RSNA Knee — Training

Thin shell: pulls `knee` from GitHub at a pinned commit, trains on the pre-mounted
competition data, exports checkpoints to `/kaggle/working`.

Current experiments: **E006a-tier-weighted-loss + E006-finetune, one run**, built
on the E005 2x2 winner (config placeholders below — the run refuses to start until
they're filled from the 2x2 result). Order of operations:

1. **E006a** (minutes): weighted-vs-unweighted `blended-cv` A/B from the attached
   E005 feature bank — zero decode. The winner's loss carries into the fine-tune.
2. **Paired frozen baseline** (minutes): the frozen E005 winner re-scored on the
   fixed 90/10 holdout the fine-tune uses — same split, so frozen-vs-finetuned is
   attributable to training alone.
3. **E006** (the hours): pixel cache at the winning crop (~75 min), then three
   per-plane fine-tunes on 2.5D adjacent-slice triplets, staged unfreeze,
   warm-started from the winning E005 heads.

Requires the `WANDB_API_KEY` secret, the `knee-labels` and `knee-e005-artifacts`
datasets, and internet.

In [ ]:
# Pin a commit so every checkpoint traces to exact code. Training notebooks have internet.
# --no-deps everywhere: Kaggle's image already ships torch/timm/sklearn/numpy compiled
# together; letting pip resolve our pins upgrades numpy and breaks the whole stack.
# STALE for E006 — bump to the e006-notebook squash merge before `kaggle kernels push`
# (this run needs evaluate_holdout, absent at cdbf23d).
COMMIT = "cdbf23d"
%pip install -q --no-deps "git+https://github.com/Josie29/capstone-rsna-knee@{COMMIT}#egg=knee"
%pip install -q --no-deps pydicom pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg pylibjpeg-rle

import numpy  # fail fast if the image stack is broken or missing
import timm
import torch

print("numpy", numpy.__version__, "| torch", torch.__version__, "| timm", timm.__version__)

from knee.series import SeriesType
from knee.train_blended import BLENDED_LABEL_SOURCE

In [ ]:
# Competition data (DICOMs + series metadata) is pre-mounted; blended soft labels
# (with the per-cell __weight companions for E006a) via the knee-labels dataset;
# the E005 winner's feature bank + checkpoints via knee-e005-artifacts.
from pathlib import Path

from knee.data import load_blended_labels, weight_matrix

SLUG = "rsna-knee-abnormality-detection"
candidates = [Path("/kaggle/input/competitions") / SLUG, Path("/kaggle/input") / SLUG]
COMP_ROOT = next((p for p in candidates if (p / "train.csv").exists()), None)
if COMP_ROOT is None:
    listing = {str(d): [c.name for c in d.iterdir()] for d in Path("/kaggle/input").iterdir() if d.is_dir()}
    raise FileNotFoundError(f"competition data not found; mounts: {listing}")
print("competition root:", COMP_ROOT)


def find_input(name: str, filename: str) -> Path:
    bases = [Path("/kaggle/input") / name, Path("/kaggle/input/datasets/josiemachalek") / name]
    for base in bases:
        if (base / filename).exists():
            return base / filename
    listing = {str(d): [c.name for c in d.iterdir()] for d in Path("/kaggle/input").iterdir() if d.is_dir()}
    raise FileNotFoundError(f"{name}/{filename} not found; mounts: {listing}")


labels = load_blended_labels(find_input("knee-labels", "blended_labels_v1.csv"), include_weights=True)
weights = weight_matrix(labels)
print(f"blended labels: {len(labels)} studies; weights {weights.shape}")

In [ ]:
# Metrics/hyperparams only -- no report text, no StudyInstanceUIDs (rule 2.4.b).
# Best-effort: the run proceeds without wandb if the secret isn't configured.
run = None
try:
    import wandb
    from kaggle_secrets import UserSecretsClient

    wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))
    run = wandb.init(
        project="rsna-knee",
        config={"commit": COMMIT, "label_source": BLENDED_LABEL_SOURCE, "n_label_studies": len(labels)},
    )
except Exception as exc:  # noqa: BLE001 — telemetry must never kill a training run
    print(f"wandb disabled: {exc}")

In [ ]:
# E006 config. The two WINNER_* placeholders come from the E005 2x2 result and the
# guard below refuses to run until they're set — a fine-tune on the wrong geometry
# or warm start would burn hours measuring the wrong thing.
from knee.model import DEFAULT_BACKBONE, HeadType

SERIES_TYPES = [SeriesType.SAGITTAL_FLUID, SeriesType.CORONAL_FLUID, SeriesType.AXIAL_FLUID]
BACKBONE = DEFAULT_BACKBONE
INPUT_SIZE = 224

WINNER_CROP_MM: float | None = ...  # FILL from E005: 140.0 (crop140 won) or None (full_frame won)
WINNER_HEAD_TYPE: HeadType = ...  # FILL from E005: HeadType.ATTENTION or HeadType.MEAN_MAX
WINNER_BANK_FILE = ...  # FILL: the winning bank filename inside knee-e005-artifacts
if ... in (WINNER_CROP_MM, WINNER_HEAD_TYPE, WINNER_BANK_FILE):
    raise RuntimeError("Fill the WINNER_* placeholders from the E005 2x2 result before running")

CHECKPOINT_DIR = Path("/kaggle/working")
CACHE_DIR = Path("/tmp/pixel_cache")  # ephemeral: only checkpoints/banks persist as output

In [ ]:
# E006a (minutes, zero decode): weighted-vs-unweighted blended-cv A/B from the
# attached E005 bank — identical bank/folds/head, only the loss weighting changes.
# The winner's loss carries into the fine-tune below (margin: repeat spread).
from knee.cv import cross_validate, load_feature_bank

bank = load_feature_bank(find_input("knee-e005-artifacts", WINNER_BANK_FILE))
cv_unweighted = cross_validate(bank, head_type=WINNER_HEAD_TYPE)
cv_weighted = cross_validate(bank, head_type=WINNER_HEAD_TYPE, cell_weights=weights)
for name, cv in (("unweighted", cv_unweighted), ("tier-weighted", cv_weighted)):
    print(f"{name}: macro OOF AUC {cv.macro_auc:.3f} "
          f"({', '.join(f'{m:.3f}' for m in cv.macro_auc_per_repeat)})")

spread = max(cv_unweighted.macro_auc_per_repeat) - min(cv_unweighted.macro_auc_per_repeat)
USE_WEIGHTS = cv_weighted.macro_auc > cv_unweighted.macro_auc + spread
print(f"E006a verdict: fine-tune trains {'WEIGHTED' if USE_WEIGHTS else 'UNWEIGHTED'}")

In [ ]:
# Fixed 90/10 split (the E006 regime marker) + the paired frozen baseline: the
# E005 winner's head family re-fit on the training side of THIS split and scored
# on its validation side — so frozen-vs-finetuned differs only in training.
import numpy as np

from knee.cv import evaluate_holdout, stratified_holdout
from knee.labels import LABEL_COLUMNS

label_matrix = labels[list(LABEL_COLUMNS)].to_numpy(dtype=np.float32)
val_mask = stratified_holdout(label_matrix, val_fraction=0.1, seed=0)
print(f"split: {int((~val_mask).sum())} train / {int(val_mask.sum())} val")

frozen_val = evaluate_holdout(
    bank, val_mask, head_type=WINNER_HEAD_TYPE, cell_weights=weights if USE_WEIGHTS else None
)
frozen_macro = float(np.nanmean(list(frozen_val.values())))
print(f"frozen baseline val macro AUC: {frozen_macro:.3f}")
print({label: round(auc, 3) for label, auc in frozen_val.items()})

In [ ]:
# E006 (the hours): pixel cache at the winning geometry (~75 min decode), then
# three per-plane fine-tunes (~30-40 min each on the T4). Heads warm-start from
# the winning E005 checkpoints (attached flat in knee-e005-artifacts); the loss is
# whatever E006a's verdict picked. After the loop, the fine-tuned ENSEMBLE scores
# on the same split — the apples-to-apples number against frozen_macro.
from knee.finetune import (
    FinetuneConfig,
    build_pixel_cache,
    evaluate_ensemble_holdout,
    finetune_plane,
)
from knee.model import InputMode, KneeModel, load_model

cache = build_pixel_cache(
    COMP_ROOT, labels, CACHE_DIR, series_types=SERIES_TYPES,
    input_size=INPUT_SIZE, crop_mm=WINNER_CROP_MM,
)
print("cache coverage:", {t.value: n for t, n in cache.coverage.items()})

finetune_results, finetuned_models = {}, {}
for series_type in SERIES_TYPES:
    model = KneeModel(BACKBONE, head_type=WINNER_HEAD_TYPE, input_mode=InputMode.TRIPLETS)
    warm = load_model(find_input("knee-e005-artifacts", f"{BLENDED_LABEL_SOURCE}_{series_type.value}.pt"))
    model.head.load_state_dict(warm.model.head.state_dict())
    result = finetune_plane(
        CACHE_DIR, label_matrix, val_mask,
        series_type=series_type, model=model,
        out_path=CHECKPOINT_DIR / f"finetune_{series_type.value}.pt",
        config=FinetuneConfig(),
        input_size=INPUT_SIZE, crop_mm=WINNER_CROP_MM,
        cell_weights=weights if USE_WEIGHTS else None,
        label_source=BLENDED_LABEL_SOURCE,
    )
    finetune_results[series_type] = result
    finetuned_models[series_type] = load_model(result.checkpoint_path).model
    print(f"{series_type.value}: best per-plane val macro {result.best_val_macro_auc:.3f} (epoch {result.best_epoch + 1})")

ensemble_val = evaluate_ensemble_holdout(CACHE_DIR, finetuned_models, label_matrix, val_mask)
ensemble_macro = float(np.nanmean(list(ensemble_val.values())))
print(f"FINETUNED ensemble val macro {ensemble_macro:.3f} vs FROZEN {frozen_macro:.3f}")
print({label: round(auc, 3) for label, auc in ensemble_val.items()})

In [ ]:
# The three finetune_*.pt checkpoints persist as notebook output. Submission gate
# (team policy after E004's CV gain evaporated at the LB): only publish to
# knee-weights + run inference if the finetuned ensemble beats the frozen paired
# baseline decisively AND clears the best CV ever recorded (0.783) — otherwise
# record the numbers in experiments.md and bank the learning without a scoring run.
import math

if run is not None:
    run.config.update({
        "backbone": BACKBONE, "input_size": INPUT_SIZE,
        "crop_mm": WINNER_CROP_MM, "head_type": WINNER_HEAD_TYPE.value,
        "tier_weighted": bool(USE_WEIGHTS),
    })
    wandb.log({
        "e006a/macro_auc/unweighted": cv_unweighted.macro_auc,
        "e006a/macro_auc/weighted": cv_weighted.macro_auc,
        "holdout/frozen_macro": frozen_macro,
        "holdout/finetuned_macro": ensemble_macro,
        **{f"holdout/finetuned/{label}": auc for label, auc in ensemble_val.items() if not math.isnan(auc)},
        **{f"holdout/frozen/{label}": auc for label, auc in frozen_val.items() if not math.isnan(auc)},
        **{
            f"finetune/{series_type.value}/best_val_macro": result.best_val_macro_auc
            for series_type, result in finetune_results.items()
        },
    })
    run.finish()